In [103]:
import pandas
df= pandas.read_csv("ridge_predictions_with_price_stats.csv")

In [104]:
df.head()

,bgg_id,pred_avg_quality,mean_of_mean,max_of_max,min_of_min
0,1,0.666448,116.736489,299.95,55.97
1,10,0.711645,74.950000,74.95,74.95
2,10007,0.580688,18.519123,19.95,14.36
3,100172,0.625265,40.052193,55.95,17.96
4,100275,0.650094,44.280024,64.95,11.99


In [105]:
import json
import os
from collections import defaultdict

def load_jsonl_dir(directory):
    records = []
    
    for filename in os.listdir(directory):
        if filename.endswith(".jsonl"):
            path = os.path.join(directory, filename)
            
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        records.append(json.loads(line))
    
    return records

def aggregate_by_bgg(records):
    agg = defaultdict(list)
    
    for r in records:
        bgg_id = r.get("bgg_id")
        if bgg_id is not None:
            agg[bgg_id].append(r)
    
    return agg

def summarize(agg):
    summary = {}
    
    for bgg_id, items in agg.items():
        quality_scores = [x["quality_score"] for x in items if x.get("quality_score") is not None]
        
        sentiments = [x["sentiment"] for x in items if x.get("sentiment")]
        
        sentiment_counts = {
            "positive": sentiments.count("positive"),
            "neutral": sentiments.count("neutral"),
            "negative": sentiments.count("negative"),
        }
        
        summary[bgg_id] = {
            "count": len(items),
            "avg_quality": sum(quality_scores) / (10*len(quality_scores)) if quality_scores else None,
            "sentiments": sentiment_counts,
        }
    
    return summary

In [106]:
records = load_jsonl_dir("/Users/vishalsankarram/dsci558-project/game_forum")
agg = aggregate_by_bgg(records)
summary = summarize(agg)

In [107]:
print(len(summary))
summary[300580]

16477


{'count': 20,
 'avg_quality': 0.86,
 'sentiments': {'positive': 20, 'neutral': 0, 'negative': 0}}

In [108]:
import json
import pandas as pd
from collections import defaultdict

file_path = "bgq_reviews_bgg_linked.jsonl"

data = defaultdict(lambda: {"quality": [], "review": []})

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        
        r = json.loads(line)
        bgg_id = r.get("review_game_bgg_id")

        if bgg_id is None:
            continue

        # quality_score (out of 10)
        if r.get("quality_score") is not None:
            data[bgg_id]["quality"].append(r["quality_score"])

        # review_score (out of 5, handle "4 Stars")
        score = r.get("review_score")
        try:
            if score not in [None, ""]:
                data[bgg_id]["review"].append(
                    float(str(score).replace(" Stars", ""))
                )
        except:
            pass


# -----------------------------
# Build DataFrame
# -----------------------------
rows = []

for bgg_id, vals in data.items():
    q = vals["quality"]
    r = vals["review"]

    avg_q = sum(q) / len(q) if q else None
    avg_r = sum(r) / len(r) if r else None

    rows.append({
        "bgg_id": bgg_id,
        "quality_norm": avg_q / 10 if avg_q is not None else None,
        "review_norm": avg_r / 5 if avg_r is not None else None
    })

df_final = pd.DataFrame(rows)

In [109]:
print(df_final)

      bgg_id  quality_norm  review_norm
0       6137      0.475000     0.800000
1     300580      0.675000     1.000000
2     295486      0.366667     0.816667
3     122522      0.200000     0.500000
4     328908      0.900000     0.900000
...      ...           ...          ...
1042  318472      0.500000     1.000000
1043  339532      0.600000          NaN
1044    4098      0.500000     1.000000
1045  219650      0.700000     1.000000
1046   68448      0.500000     1.000000

[1047 rows x 3 columns]


In [110]:
df_final["summary_avg_quality"] = df_final["bgg_id"].map(
    lambda x: summary.get(x, {}).get("avg_quality")
)

In [111]:
df_final = df_final.dropna()

In [112]:
print(df_final)

      bgg_id  quality_norm  review_norm  summary_avg_quality
1     300580      0.675000     1.000000             0.860000
3     122522      0.200000     0.500000             0.783721
4     328908      0.900000     0.900000             0.759459
5     290448      0.816667     0.966667             0.828571
6     120605      0.640000     0.500000             0.800000
...      ...           ...          ...                  ...
1041     822      0.500000     1.000000             0.767513
1042  318472      0.500000     1.000000             0.900000
1044    4098      0.500000     1.000000             0.794231
1045  219650      0.700000     1.000000             0.800000
1046   68448      0.500000     1.000000             0.782752

[723 rows x 4 columns]


In [113]:
df_final = df_final.merge(
    df[["bgg_id", "pred_avg_quality"]],
    on="bgg_id",
    how="left"
)

In [114]:
df_final = df_final.dropna(subset=[
    "review_norm", "quality_norm", "summary_avg_quality", "pred_avg_quality"
])

In [100]:
df_final = df_final.merge(
    df[["bgg_id", "pred_avg_quality"]],
    on="bgg_id",
    how="left"
)
df_final = df_final.dropna()

In [123]:
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df_final[["review_norm", "quality_norm", "summary_avg_quality"]].values
y = df_final["pred_avg_quality"].values

def loss(w):
    y_pred = X @ w
    return np.mean((y - y_pred) ** 2)

# constraint: weights sum to 1
constraints = {
    'type': 'eq',
    'fun': lambda w: np.sum(w) - 1
}

# bounds: each weight >= 0.1
bounds = [(0.05, 1), (0.05, 1), (0.1, 1)]

# initial guess (must satisfy sum=1 and >=0.1)
w0 = np.array([0.2, 0.2, 0.6])

result = minimize(loss, w0, bounds=bounds, constraints=constraints)

weights = result.x

print("Weights (>= 0.1):")
print(f"review_norm: {weights[0]}")
print(f"quality_norm: {weights[1]}")
print(f"summary_avg_quality: {weights[2]}")
print("Sum:", weights.sum())

# evaluate
y_pred = X @ weights

mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print("\nMetrics:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

Weights (>= 0.1):
review_norm: 0.05
quality_norm: 0.18929142775065674
summary_avg_quality: 0.7607085722493432
Sum: 1.0

Metrics:
MAE: 0.06583305343201264
RMSE: 0.08315878674349762
R2: -4.262404777948882


In [124]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

df_final["final_score"] = df_final["review_norm"] * 0.05 + df_final["quality_norm"] * 0.2 + df_final["summary_avg_quality"] * 0.75
y_true = df_final["final_score"]
y_pred = df_final["pred_avg_quality"]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.06575947740325802
RMSE: 0.08322470910443469
R2: -0.3065185036100233
